# StandUp4AI Training
XLM-RoBERTa-base for word-level laughter detection

**Target:** French, English, Spanish F1 >= 0.70

In [ ]:
!pip install -q torch transformers scikit-learn pandas numpy requests

In [ ]:
import torchimport pandas as pdimport numpy as npimport requestsimport iofrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import f1_scorefrom transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmupfrom torch.optim import AdamWfrom torch.utils.data import Dataset, DataLoaderimport warningswarnings.filterwarnings('ignore')# ConfigMODEL_NAME = 'xlm-roberta-base'MAX_LEN = 64BATCH_SIZE = 32EPOCHS = 5LR = 2e-5DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {DEVICE}')

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osOUTPUT_DIR = '/content/drive/MyDrive/standup4ai_models'os.makedirs(OUTPUT_DIR, exist_ok=True)print(f'Output: {OUTPUT_DIR}')

In [ ]:
BASE = 'https://raw.githubusercontent.com/Das-rebel/standup4ai-colab/main'FILES = {'-1FrUOEswOk.csv': 'fr', '0g7nezWZyfY.csv': 'en', '1xvwYZwm8Ig.csv': 'en', '6JQzl2LlXbQ.csv': 'es'}dfs = []for fname, lang in FILES.items():    url = f'{BASE}/{fname}'    print(f'Fetching {fname}...')    r = requests.get(url)    df = pd.read_csv(io.StringIO(r.text))    df['language'] = lang    dfs.append(df)    print(f'  -> {len(df)} rows')data = pd.concat(dfs, ignore_index=True)print(f'Total: {len(data)} samples')

In [ ]:
data['label'] = data['label'].map({'L': 1, 'O': 0})print(f'NaN: {data["label"].isna().sum()}')print(f'Distribution:\n{data["label"].value_counts()}')

In [ ]:
train_dfs, val_dfs = [], []for lang in data['language'].unique():    ld = data[data['language'] == lang]    tr, vl = train_test_split(ld, test_size=0.2, random_state=42, stratify=ld['label'])    train_dfs.append(tr)    val_dfs.append(vl)    print(f'{lang}: train={len(tr)}, val={len(vl)}')train_data = pd.concat(train_dfs).reset_index(drop=True)val_data = pd.concat(val_dfs).reset_index(drop=True)print(f'Total: train={len(train_data)}, val={len(val_data)}')

In [ ]:
class LaughterDS(Dataset):    def __init__(self, texts, labels, tok, max_len):        self.texts, self.labels, self.tok, self.max_len = texts, labels, tok, max_len    def __len__(self): return len(self.texts)    def __getitem__(self, idx):        enc = self.tok(str(self.texts[idx]), add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'label': torch.tensor(self.labels[idx], dtype=torch.long)}

In [ ]:
class XLMR(torch.nn.Module):    def __init__(self, mname):        super().__init__()        self.enc = AutoModel.from_pretrained(mname)        self.drop = torch.nn.Dropout(0.3)        self.cls = torch.nn.Linear(self.enc.config.hidden_size, 2)    def forward(self, ids, mask):        out = self.enc(input_ids=ids, attention_mask=mask)        return self.cls(self.drop(out.last_hidden_state[:, 0]))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)train_ds = LaughterDS(train_data['text'].values, train_data['label'].values, tokenizer, MAX_LEN)val_ds = LaughterDS(val_data['text'].values, val_data['label'].values, tokenizer, MAX_LEN)train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE)print(f'Train: {len(train_ld)}, Val: {len(val_ld)}')model = XLMR(MODEL_NAME).to(DEVICE)opt = AdamW(model.parameters(), lr=LR, weight_decay=0.01)sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*len(train_ld)*EPOCHS), num_training_steps=len(train_ld)*EPOCHS)print('Ready!')

In [ ]:
def train_ep(m, ld, opt, sch):    m.train()    tot, preds, labs = 0, [], []    crit = torch.nn.CrossEntropyLoss()    for b in ld:        opt.zero_grad()        logits = m(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))        loss = crit(logits, b['label'].to(DEVICE))        loss.backward()        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)        opt.step()        sch.step()        tot += loss.item()        preds.extend(torch.argmax(logits, 1).cpu().numpy())        labs.extend(b['label'].numpy())    return tot/len(ld), f1_score(labs, preds, average='weighted')

In [ ]:
def eval_ep(m, ld):    m.eval()    tot, preds, labs = 0, [], []    crit = torch.nn.CrossEntropyLoss()    with torch.no_grad():        for b in ld:            logits = m(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))            loss = crit(logits, b['label'].to(DEVICE))            tot += loss.item()            preds.extend(torch.argmax(logits, 1).cpu().numpy())            labs.extend(b['label'].numpy())    return tot/len(ld), f1_score(labs, preds, average='weighted'), preds, labs

In [ ]:
best_f1 = 0hist = {'tl': [], 'tf': [], 'vl': [], 'vf': []}print('Training...')for ep in range(EPOCHS):    tl, tf = train_ep(model, train_ld, opt, sched)    vl, vf, _, _ = eval_ep(model, val_ld)    hist['tl'].append(tl)    hist['tf'].append(tf)    hist['vl'].append(vl)    hist['vf'].append(vf)    print(f'Ep{ep+1}: TL={tl:.4f} TF={tf:.4f} VL={vl:.4f} VF={vf:.4f}')    if vf > best_f1:        best_f1 = vf        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best.pt')        print(f'  -> Saved F1={best_f1:.4f}')print(f'Best F1: {best_f1:.4f}')

In [ ]:
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best.pt'))_, _, preds, actual = eval_ep(model, val_ld)val_eval = val_data.copy()val_eval['pred'] = predsprint('='*50)print('RESULTS')print('='*50)for lang in ['fr', 'en', 'es']:    ld = val_eval[val_eval['language'] == lang]    if len(ld) > 0:        f1 = f1_score(ld['label'], ld['pred'], average='weighted')        st = 'PASS' if f1 >= 0.70 else 'FAIL'        print(f'{lang.upper()}: F1={f1:.4f} [{st}]')overall = f1_score(actual, preds, average='weighted')st = 'PASS' if overall >= 0.70 else 'FAIL'print(f'OVERALL: F1={overall:.4f} [{st}]')

In [ ]:
import jsonmodel.save_pretrained(f'{OUTPUT_DIR}/xlmr')tokenizer.save_pretrained(f'{OUTPUT_DIR}/xlmr')with open(f'{OUTPUT_DIR}/results.json', 'w') as f:    json.dump({'overall': overall, 'best': best_f1}, f)print('Done!')

## Done! ✅

Model saved to `/content/drive/MyDrive/standup4ai_models/`